# GECKO 2 (Rust + CUDA/WebGPU) — Paper Benchmark Suite (Google Colab T4)

Reproduces the **exact datasets from the original GECKO paper** (*Torreno & Trelles, BMC Bioinformatics 2015, 16:250*) and extends them to large-scale workloads that were previously impractical on Legacy GECKO v1 — all in **~35–45 minutes on a single free Colab T4 GPU**.

### Experiments Included (all checkpointed to JSON with `RESUME = True`)
1. **Experiment 1 — Direct Replication of Torreno & Trelles (2015) Table 2 (All 5 Backends: `legacy_v1`, `c_fast`, `cpu`, `wgpu`, `cuda`):**
   Exact 5 pairwise benchmarks spanning 5 orders of magnitude (`4 kb → 249 Mb`), downloaded by GenBank/RefSeq accession:
   - *TYLCV* (`AM409201.1` × `EU085423.2`, 4 kb)
   - *Buchnera aphidicola* (`NC_002528.1` × `NC_011833.1`, 0.64 Mb)
   - *E. coli* K-12 × O157:H7 (`NC_000913.2` × `NC_002695.1`, 4.6–5.4 Mb)
   - *Drosophila* `2R` × `3` (`NT_033778.3` × `NC_009006.2`, 20–21 Mb)
   - *H. sapiens* chr1 × *P. troglodytes* chr1 (`NC_000001.11` × `NC_006468.3`, 249 Mb × 228 Mb)
   *(Est. ~15–20 min total; >90% of time is `legacy_v1` on `HS×PT chr1`.)*
2. **Experiment 2 — Breaking the Next Barrier: Original 2015 Table 4 All-vs-All 10-Species Mammalian Chr1 (45 Pairs):**
   All 10 mammalian chr1 accessions from Table 1 (`HS, PT, MM, PA, GG, MMU, RN, BT, CF, SS`, 121–312 Mb each). In 2015 this took **39.7 hours** on GECKO v1 (and **6.5–55 days** on MUMmer/Gepard/LASTZ). Here GECKO v2 runs **all 45 pairs on CUDA in ~4–6 min** (+ all 9 `HS × species` pairs from Fig. 2 on Rust `cpu` in ~3 min).
3. **Experiment 3 — Whole-Genome 24-Chromosome Human × Chimp (`GRCh38.p14 × Clint_PTRv2`, ~3.1 Gb × 2.9 Gb):**
   Fast in-memory backends only (`c_fast`, `cpu`, `wgpu`, `cuda` — no slow `legacy_v1`), demonstrating full mammalian whole-genome comparison in **~90 s on T4 CUDA**.
4. **Experiment 4 — Paper Ablations:**
   - **4A:** Zero-Spill Reference-Slab Memory Scaling on `HS chr1 × PT chr1` (`-m 512M, 1G, 2G, 4G`)
   - **4B:** Dense Short-Seed & Frequency Frontier (`WL ∈ {12,16,24,32} × FL ∈ {20,100,1000}`) on *Drosophila* `2R × 3`
5. **Cell 9:** Generates publication-grade 300-DPI PNG + PDF figures, LaTeX tables, and bundles **everything** into `/content/gecko2_paper_results.zip` for 1-click download.

> **Before running:** Set `Runtime → Change runtime type → T4 GPU`.


In [ ]:
# 1. Capture hardware & toolchain provenance (saved for the paper's Experimental Setup section)
import json, os, platform, subprocess

def cmd_out(cmd):
    try:
        return subprocess.check_output(cmd, shell=True, stderr=subprocess.STDOUT, text=True).strip()
    except Exception as e:
        return f'unavailable ({e})'

hw = {
    'platform': platform.platform(),
    'cpu_count': os.cpu_count(),
    'lscpu': cmd_out('lscpu | grep -E "Model name|CPU\\(s\\):|Thread|Core|L[123] cache|Flags" | head -n 12'),
    'mem': cmd_out('free -h'),
    'disk': cmd_out('df -h /content'),
    'gpu': cmd_out('nvidia-smi --query-gpu=name,driver_version,memory.total,compute_cap --format=csv,noheader'),
    'nvcc': cmd_out('nvcc --version | tail -n 2'),
    'gcc': cmd_out('gcc --version | head -n 1'),
}
os.makedirs('/content/paper_results', exist_ok=True)
with open('/content/paper_results/hardware_info.json', 'w') as f:
    json.dump(hw, f, indent=2)

for k, v in hw.items():
    print(f'=== {k} ===\n{v}\n')


In [ ]:
# 2. Install Rust + Vulkan/WGPU runtime and build all 3 GECKO implementations
import os, subprocess

def sh(cmd):
    print(f'$ {cmd}', flush=True)
    subprocess.run(cmd, shell=True, check=True)

sh('apt-get update -qq && apt-get install -y -qq libvulkan1 mesa-vulkan-drivers vulkan-tools build-essential curl >/dev/null')

if not os.path.exists('/root/.cargo/bin/cargo'):
    sh('curl --proto "=https" --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y --default-toolchain stable')
os.environ['PATH'] = '/root/.cargo/bin:' + os.environ['PATH']
sh('rustc --version && cargo --version')

# 2a. GECKO v2 (Rust + CUDA + WGPU) from branch `rust-rewrite`
if not os.path.exists('/content/gecko'):
    sh('git clone --depth 1 -b rust-rewrite https://github.com/otorreno/gecko.git /content/gecko')
else:
    sh('git -C /content/gecko fetch --depth 1 origin rust-rewrite && git -C /content/gecko reset --hard origin/rust-rewrite')

cap = subprocess.check_output(
    'nvidia-smi --query-gpu=compute_cap --format=csv,noheader', shell=True, text=True
).strip().splitlines()[0].replace('.', '')
os.environ['GECKO_CUDA_ARCH'] = f'-gencode arch=compute_{cap},code=sm_{cap}'
print(f"Building GECKO v2 (Rust + CUDA) with GECKO_CUDA_ARCH={os.environ['GECKO_CUDA_ARCH']}")
sh('cd /content/gecko && cargo build --release --bin gecko --features cuda')

GECKO_BIN = '/content/gecko/target/release/gecko'
sh(f'{GECKO_BIN} backends')

# 2b. Intermediate C in-memory baseline (`c_fast`, -O3 -march=native) from branch `gpu-backends`
if not os.path.exists('/content/gecko_c_fast'):
    sh('git clone --depth 1 -b gpu-backends https://github.com/otorreno/gecko.git /content/gecko_c_fast')
else:
    sh('git -C /content/gecko_c_fast fetch --depth 1 origin gpu-backends && git -C /content/gecko_c_fast reset --hard origin/gpu-backends')
sh('make -C /content/gecko_c_fast/src gecko_fast')
C_FAST_BIN = '/content/gecko_c_fast/bin/gecko_fast'
assert os.path.exists(C_FAST_BIN), 'Failed to build gecko_fast'

# 2c. Original published Legacy GECKO v1 (Torreno & Trelles 2015) from branch `master`
if not os.path.exists('/content/gecko_legacy_v1'):
    sh('git clone --depth 1 -b master https://github.com/otorreno/gecko.git /content/gecko_legacy_v1')
else:
    sh('git -C /content/gecko_legacy_v1 fetch --depth 1 origin master && git -C /content/gecko_legacy_v1 reset --hard origin/master')
sh('make -C /content/gecko_legacy_v1/src all')
LEGACY_V1_DIR = '/content/gecko_legacy_v1/bin'
for b in ['reverseComplement', 'words', 'sortWords', 'w2hd', 'hits', 'sortHits', 'filterHits', 'FragHits', 'combineFrags']:
    assert os.path.exists(os.path.join(LEGACY_V1_DIR, b)), f'Missing legacy v1 binary: {b}'

print('\nAll 3 GECKO builds ready!')
print('  Rust v2   :', GECKO_BIN)
print('  C fast    :', C_FAST_BIN)
print('  Legacy v1 :', LEGACY_V1_DIR)


In [ ]:
# 3. Master Benchmark Configuration & Verified NCBI Chunked Downloader
import json, os, shutil, subprocess, time, urllib.parse, urllib.request

# --- Experiment Toggles (all checkpointed; safe to re-run with RESUME=True) ---
RUN_EXP1_ORIGINAL_PAIRWISE = True   # Exact 5 pairs from Torreno & Trelles (2015) Table 2 (ALL 5 backends, ~15-20 min)
RUN_EXP2_ALL_VS_ALL_45     = True   # Exact 10 mammalian chr1s from 2015 Table 4 (45 pairs on CUDA + 9 HS pairs on CPU, ~8 min)
RUN_EXP2_ALL_45_ON_CPU     = False  # Set True to also run all 45 pairs on Rust CPU (+15-20 min)
RUN_EXP3_WHOLE_GENOME_24CHR = True  # All 24 Human vs Chimp chrs on fast backends ONLY (no legacy_v1, ~8-12 min)
RUN_EXP4_ABLATIONS         = True   # 4A: Slab memory (-m 512M..4G) on HS×PT chr1; 4B: (WL, FL) matrix on Drosophila (~5 min)

RESUME = True
CHECK_PARITY = True
MEMORY_BUDGET = '4G'
DEFAULT_FL = 20

EXP1_BACKENDS = ['cpu', 'cuda', 'wgpu', 'c_fast', 'legacy_v1']
EXP3_BACKENDS = ['cpu', 'cuda', 'c_fast', 'wgpu']  # Fast in-memory backends only

DATA_DIR = '/content/gecko_paper_data'
WORK_DIR = '/content/gecko_paper_work'
OUT_DIR  = '/content/paper_results'
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(WORK_DIR, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)

# Exact Table 1 accessions from Torreno & Trelles (BMC Bioinformatics 2015, 16:250)
ORIG_PAIRWISE_BENCHMARKS = [
    {
        'id': 'tylcv',
        'label': 'TYLCV vs TYLCV-lr2 (Virus, 4 kb)',
        'q_name': 'TYLCV', 'q_acc': 'AM409201.1',
        'r_name': 'TYLCV-lr2', 'r_acc': 'EU085423.2',
        'wl': 12, 'l': 40, 's': 50, 'fl': DEFAULT_FL,
        'pub_2015_s': {'GECKO_v1_2015': 0.36, 'Gepard': 0.84, 'MUMmer': 0.00, 'Mauve': 0.06, 'LASTZ': 0.04, 'LAST': 0.00},
    },
    {
        'id': 'buchnera',
        'label': 'Buchnera APS vs 5A (Bacteria, 0.64 Mb)',
        'q_name': 'Buchnera_APS', 'q_acc': 'NC_002528.1',
        'r_name': 'Buchnera_5A',  'r_acc': 'NC_011833.1',
        'wl': 12, 'l': 40, 's': 50, 'fl': DEFAULT_FL,
        'pub_2015_s': {'GECKO_v1_2015': 1.60, 'Gepard': 2.56, 'MUMmer': 0.44, 'Mauve': 6.73, 'LASTZ': 0.46, 'LAST': 46.20},
    },
    {
        'id': 'ecoli',
        'label': 'E. coli K-12 vs O157:H7 (Bacteria, 5 Mb)',
        'q_name': 'Ecoli_K12',  'q_acc': 'NC_000913.2',
        'r_name': 'Ecoli_O157', 'r_acc': 'NC_002695.1',
        'wl': 12, 'l': 40, 's': 50, 'fl': DEFAULT_FL,
        'pub_2015_s': {'GECKO_v1_2015': 17.20, 'Gepard': 33.12, 'MUMmer': 10.63, 'Mauve': 45.92, 'LASTZ': 1.83, 'LAST': 109.00},
    },
    {
        'id': 'drosophila',
        'label': 'D. mel 2R vs D. pse 3 (Fly, 20 Mb)',
        'q_name': 'Dmel_2R', 'q_acc': 'NT_033778.3',
        'r_name': 'Dpse_3',  'r_acc': 'NC_009006.2',
        'wl': 16, 'l': 100, 's': 50, 'fl': DEFAULT_FL,
        'pub_2015_s': {'GECKO_v1_2015': 48.72, 'Gepard': 238.34, 'MUMmer': 45.99, 'Mauve': 294.92, 'LASTZ': 19.64, 'LAST': 1593.00},
    },
    {
        'id': 'hs_pt_chr1',
        'label': 'H. sapiens chr1 vs P. troglodytes chr1 (249 Mb)',
        'q_name': 'HS_chr1', 'q_acc': 'NC_000001.11',
        'r_name': 'PT_chr1', 'r_acc': 'NC_006468.3',
        'wl': 32, 'l': 1000, 's': 75, 'fl': DEFAULT_FL,
        'pub_2015_s': {'GECKO_v1_2015': 11848.15, 'Gepard': 7084.00, 'MUMmer': 23226.00, 'Mauve': 604800.0, 'LASTZ': 78360.00, 'LAST': None},
    },
]

# Exact 10 Mammalian Chromosome 1 accessions from Torreno & Trelles (2015) Table 1 & Table 4
MAMMALIAN_CHR1_10 = [
    ('HS',  'Homo sapiens',          'NC_000001.11'),
    ('PT',  'Pan troglodytes',       'NC_006468.3'),
    ('MM',  'Macaca mulatta',        'NC_007858.1'),
    ('PA',  'Pongo abelii',          'NC_012591.1'),
    ('GG',  'Gorilla gorilla',       'NC_018424.1'),
    ('MMU', 'Mus musculus',          'NC_000067.6'),
    ('RN',  'Rattus norvegicus',     'NC_005100.3'),
    ('BT',  'Bos taurus',            'AC_000158.1'),
    ('CF',  'Canis lupus familiaris','NC_006583.3'),
    ('SS',  'Sus scrofa',            'NC_010443.4'),
]

# Published 2015 Table 4 totals across all 45 pairs (seconds)
PUB_2015_TABLE4_TOTALS = {
    'GECKO_v1_2015_cached':   142954.0,
    'GECKO_v1_2015_nocache':  318591.0,
    'MUMmer_2015':            558360.0,
    'Gepard_2015':            576889.0,
    'LASTZ_2015':            4752315.0,
}

# Published 2015 Table 4 pairwise GECKO v1 times (seconds) for HS vs other 9 species
PUB_2015_TABLE4_HS_ROW = {
    'PT': 19190, 'MM': 2438, 'PA': 11282, 'GG': 11433,
    'MMU': 9358, 'RN': 11367, 'BT': 3768, 'CF': 2944, 'SS': 5875,
}

def _http_get_text(url, retries=6, timeout=120):
    last_err = None
    for attempt in range(retries):
        try:
            req = urllib.request.Request(url, headers={'User-Agent': 'gecko2-benchmark/1.0'})
            with urllib.request.urlopen(req, timeout=timeout) as resp:
                return resp.read().decode('utf-8', errors='replace')
        except Exception as e:
            last_err = e
            wait = min(30, 2 ** attempt)
            print(f'    retry {attempt+1}/{retries} after error ({e}), sleeping {wait}s...', flush=True)
            time.sleep(wait)
    raise RuntimeError(f'Failed GET {url}: {last_err}')

def ncbi_seq_len(acc):
    url = f'https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esummary.fcgi?db=nuccore&id={urllib.parse.quote(acc)}&retmode=json'
    data = json.loads(_http_get_text(url))
    uids = data.get('result', {}).get('uids', [])
    if not uids:
        raise RuntimeError(f'Accession not found in NCBI nuccore: {acc}')
    return int(data['result'][uids[0]]['slen'])

def fetch_ncbi_fasta(acc, out_fa, chunk_bp=25_000_000):
    '''Download exact NCBI nuccore accession in verified uppercase FASTA (chunked for large chromosomes).'''
    meta_path = out_fa + '.meta.json'
    if os.path.exists(out_fa) and os.path.exists(meta_path):
        meta = json.load(open(meta_path))
        if meta.get('acc') == acc and meta.get('bases', 0) > 0:
            return out_fa, meta['bases']

    slen = ncbi_seq_len(acc)
    tmp_fa = out_fa + '.tmp'
    written_bases = 0
    t0 = time.perf_counter()
    print(f'  Downloading {acc} ({slen/1e6:.3f} Mbp)...', end=' ', flush=True)

    with open(tmp_fa, 'w') as fout:
        fout.write(f'>{acc}\n')
        if slen <= chunk_bp:
            url = f'https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi?db=nuccore&id={urllib.parse.quote(acc)}&rettype=fasta&retmode=text'
            txt = _http_get_text(url)
            for line in txt.splitlines():
                if not line or line.startswith('>'):
                    continue
                s = line.strip().upper()
                fout.write(s + '\n')
                written_bases += len(s)
            time.sleep(0.35)
        else:
            for start in range(1, slen + 1, chunk_bp):
                stop = min(slen, start + chunk_bp - 1)
                url = (
                    f'https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi?db=nuccore'
                    f'&id={urllib.parse.quote(acc)}&rettype=fasta&retmode=text'
                    f'&seq_start={start}&seq_stop={stop}'
                )
                txt = _http_get_text(url, timeout=180)
                chunk_len = 0
                for line in txt.splitlines():
                    if not line or line.startswith('>'):
                        continue
                    s = line.strip().upper()
                    fout.write(s + '\n')
                    chunk_len += len(s)
                expected = stop - start + 1
                if chunk_len != expected:
                    raise RuntimeError(f'{acc} [{start}..{stop}]: expected {expected} bp, got {chunk_len} bp')
                written_bases += chunk_len
                time.sleep(0.35)

    if written_bases != slen:
        os.remove(tmp_fa)
        raise RuntimeError(f'{acc}: total downloaded {written_bases} bp != esummary slen {slen} bp')

    os.replace(tmp_fa, out_fa)
    with open(meta_path, 'w') as mf:
        json.dump({'acc': acc, 'bases': written_bases}, mf)
    print(f'OK ({written_bases:,} bp in {time.perf_counter()-t0:.1f}s)', flush=True)
    return out_fa, written_bases

print('Config & NCBI chunked downloader initialized.')
print(f'  Exp 1 (Original 2015 Table 2, 5 pairs × 5 backends): {RUN_EXP1_ORIGINAL_PAIRWISE}')
print(f'  Exp 2 (Original 2015 Table 4, 10 mammals = 45 pairs): {RUN_EXP2_ALL_VS_ALL_45}')
print(f'  Exp 3 (Whole-genome 24-chr Human×Chimp, fast only)  : {RUN_EXP3_WHOLE_GENOME_24CHR}')
print(f'  Exp 4 (Ablations: slab memory + WL/FL sensitivity)  : {RUN_EXP4_ABLATIONS}')


In [ ]:
# 4. Unified Benchmark Harness (legacy_v1, c_fast, cpu, wgpu, cuda + bidirectional parity + disk/RSS)
import gzip, json, os, re, resource, shutil, struct, subprocess, tempfile, time
from concurrent.futures import ThreadPoolExecutor

def parse_frags(path):
    frags = []
    with open(path, 'rb') as f:
        hdr = f.read(16)
        if len(hdr) < 16:
            return frags
        n0, n1 = struct.unpack('>QQ', hdr)
        while len(b := f.read(93)) == 93:
            diag, xs, ys, xe, ye, length, ident, score, sim, sx, sy, blk, st = struct.unpack('>qQQQQQQQfQQqB', b)
            ch = chr(st)
            if ch == 'r':
                ys = n1 - ys - 1
                ye = n1 - ye - 1
            frags.append((xs, ys, xe, ye, ch, length, score, ident, sx, sy))
    return frags

def compute_parity(ref_frags, test_frags):
    sr, st = set(ref_frags), set(test_frags)
    if not sr:
        return (100.0, 100.0) if not st else (0.0, 0.0)
    exact = len(sr & st)
    by_diag = {}
    for fr in test_frags:
        dk = (fr[4], fr[0] - fr[1] if fr[4] == 'f' else fr[0] + fr[1], fr[8], fr[9])
        by_diag.setdefault(dk, []).append((fr[0], fr[2]))
    cov = 0
    for fr in (sr - st):
        dk = (fr[4], fr[0] - fr[1] if fr[4] == 'f' else fr[0] + fr[1], fr[8], fr[9])
        for a, b in by_diag.get(dk, ()):
            if min(fr[2], b) - max(fr[0], a) + 1 >= 0.9 * (fr[2] - fr[0] + 1):
                cov += 1
                break
    return 100.0 * exact / len(sr), 100.0 * (exact + cov) / len(sr)

def _raise_stack_for_v1():
    # Legacy reverseComplement.c allocates `long offset[1_000_000]` (8 MiB) on the stack.
    soft, hard = resource.getrlimit(resource.RLIMIT_STACK)
    want = 64 * 1024 * 1024
    if soft != resource.RLIM_INFINITY and soft < want:
        target = want if hard == resource.RLIM_INFINITY else min(want, hard)
        resource.setrlimit(resource.RLIMIT_STACK, (target, hard))

def run_legacy_v1(tag, q_fa, r_fa, wl, min_len, min_sim, fl, check_parity=True):
    outdir = os.path.join(WORK_DIR, f'out_{tag}_legacy_v1')
    shutil.rmtree(outdir, ignore_errors=True)
    os.makedirs(outdir, exist_ok=True)
    n_workers = min(2, os.cpu_count() or 2)
    r_rc = os.path.join(outdir, 'Yrc.fna')

    def step(cmd):
        with tempfile.TemporaryFile() as out_f, tempfile.TemporaryFile() as err_f:
            p = subprocess.Popen(cmd, cwd=outdir, preexec_fn=_raise_stack_for_v1, stdout=out_f, stderr=err_f)
            _, status, ru = os.wait4(p.pid, 0)
            if status != 0:
                out_f.seek(0); err_f.seek(0)
                msg = (err_f.read() or out_f.read()).decode('utf-8', errors='replace').strip().splitlines()
                raise RuntimeError(f"{os.path.basename(cmd[0])} failed (status={status}): {msg[-1] if msg else ''}")
            return ru.ru_maxrss / 1024.0

    def fsize_mib(name):
        p = os.path.join(outdir, name)
        return os.path.getsize(p) / (1024 * 1024) if os.path.exists(p) else 0.0

    def dir_mib():
        tot = 0
        for root, _, files in os.walk(outdir):
            for fn in files:
                try:
                    tot += os.path.getsize(os.path.join(root, fn))
                except OSError:
                    pass
        return tot / (1024 * 1024)

    peak_rss = 0.0
    peak_disk = 0.0
    total_written = 0.0
    t_wall0 = time.perf_counter()

    try:
        # 1. Reverse complement reference
        t0 = time.perf_counter()
        peak_rss = max(peak_rss, step([f'{LEGACY_V1_DIR}/reverseComplement', r_fa, r_rc]))
        revcomp_s = time.perf_counter() - t0
        total_written += fsize_mib('Yrc.fna')
        peak_disk = max(peak_disk, dir_mib())

        # 2. Build 3 disk-based dictionaries (X, Y, Yrc)
        def build_dict(seq_fa, prefix):
            rss = step([f'{LEGACY_V1_DIR}/words', seq_fa, f'{prefix}.words.unsort'])
            w_unsort = fsize_mib(f'{prefix}.words.unsort')
            d1 = dir_mib()
            rss = max(rss, step([f'{LEGACY_V1_DIR}/sortWords', '10000000', '32', f'{prefix}.words.unsort', f'{prefix}.words.sort']))
            w_sort = fsize_mib(f'{prefix}.words.sort')
            d2 = dir_mib()
            rss = max(rss, step([f'{LEGACY_V1_DIR}/w2hd', f'{prefix}.words.sort', prefix]))
            w_hd = fsize_mib(f'{prefix}.d2hW') + fsize_mib(f'{prefix}.d2hP')
            d3 = dir_mib()
            if os.path.exists(os.path.join(outdir, f'{prefix}.words.sort')):
                os.remove(os.path.join(outdir, f'{prefix}.words.sort'))
            return rss, max(d1, d2, d3), (w_unsort + w_sort + w_hd)

        t0 = time.perf_counter()
        with ThreadPoolExecutor(max_workers=n_workers) as ex:
            futs = [
                ex.submit(build_dict, q_fa, 'X'),
                ex.submit(build_dict, r_fa, 'Y'),
                ex.submit(build_dict, r_rc, 'Yrc'),
            ]
            for fut in futs:
                r, d, w = fut.result()
                peak_rss = max(peak_rss, r)
                peak_disk = max(peak_disk, d)
                total_written += w
        dict_s = time.perf_counter() - t0
        peak_disk = max(peak_disk, dir_mib())

        # 3. Forward & reverse comparisons: hits -> sortHits -> filterHits -> FragHits
        def run_strand(y_prefix, y_fa, strand):
            h = f'X-{y_prefix}.hits'
            hs = f'{h}.sorted'
            hf = f'{hs}.filtered'
            fr = f's{strand}.frags'
            rss = step([f'{LEGACY_V1_DIR}/hits', 'X', y_prefix, h, str(fl), str(wl)])
            s_h = fsize_mib(h); d1 = dir_mib()
            rss = max(rss, step([f'{LEGACY_V1_DIR}/sortHits', '10000000', '32', h, hs]))
            s_hs = fsize_mib(hs); d2 = dir_mib()
            rss = max(rss, step([f'{LEGACY_V1_DIR}/filterHits', hs, hf, str(wl)]))
            s_hf = fsize_mib(hf); d3 = dir_mib()
            for fn in (h, hs):
                p = os.path.join(outdir, fn)
                if os.path.exists(p):
                    os.remove(p)
            rss = max(rss, step([
                f'{LEGACY_V1_DIR}/FragHits', q_fa, y_fa, hf, fr,
                str(min_len), str(min_sim), str(wl), '1', strand,
            ]))
            s_fr = fsize_mib(fr); d4 = dir_mib()
            if os.path.exists(os.path.join(outdir, hf)):
                os.remove(os.path.join(outdir, hf))
            return rss, max(d1, d2, d3, d4), (s_h + s_hs + s_hf + s_fr)

        t0 = time.perf_counter()
        with ThreadPoolExecutor(max_workers=n_workers) as ex:
            f_fwd = ex.submit(run_strand, 'Y', r_fa, 'f')
            f_rev = ex.submit(run_strand, 'Yrc', r_rc, 'r')
            for fut in (f_fwd, f_rev):
                r, d, w = fut.result()
                peak_rss = max(peak_rss, r)
                peak_disk = max(peak_disk, d)
                total_written += w
        cmp_s = time.perf_counter() - t0

        # 4. Combine forward + reverse fragments
        t0 = time.perf_counter()
        peak_rss = max(peak_rss, step([f'{LEGACY_V1_DIR}/combineFrags', 'sf.frags', 'sr.frags', 'final.frags']))
        merge_s = time.perf_counter() - t0
        wall_s = time.perf_counter() - t_wall0

        final_frags = os.path.join(outdir, 'final.frags')
        frags = parse_frags(final_frags)
        n_fwd = sum(1 for fr in frags if fr[4] == 'f')
        n_rev = len(frags) - n_fwd
        with open(final_frags, 'rb') as fp:
            q_bases, r_bases = struct.unpack('>QQ', fp.read(16))

        shutil.rmtree(outdir, ignore_errors=True)
        return {
            'ok': True,
            'backend_label': f'legacy_v1 (C 2015 disk-based, {n_workers}w)',
            'wall_s': round(wall_s, 3),
            'load_s': 0.0,
            'compute_s': round(wall_s, 3),
            'write_s': round(merge_s, 3),
            'revcomp_s': round(revcomp_s, 3),
            'dict_s': round(dict_s, 3),
            'cmp_s': round(cmp_s, 3),
            'rss_mib': round(peak_rss, 1),
            'temp_disk_mib': round(peak_disk, 1),
            'total_written_mib': round(total_written, 1),
            'fragments': len(frags),
            'forward_fragments': n_fwd,
            'reverse_fragments': n_rev,
            'query_bases': q_bases,
            'reference_bases': r_bases,
        }, (frags if check_parity else None)
    except Exception as exc:
        shutil.rmtree(outdir, ignore_errors=True)
        return {'ok': False, 'error': str(exc)}, None

def run_backend(backend, tag, q_fa, r_fa, wl, min_len, min_sim, fl, mem_budget=MEMORY_BUDGET, check_parity=True):
    if backend == 'legacy_v1':
        return run_legacy_v1(tag, q_fa, r_fa, wl, min_len, min_sim, fl, check_parity=check_parity)

    outdir = os.path.join(WORK_DIR, f'out_{tag}_{backend}')
    shutil.rmtree(outdir, ignore_errors=True)
    os.makedirs(outdir, exist_ok=True)
    if backend == 'c_fast':
        prefix_path = os.path.join(outdir, tag)
        threads = str(os.cpu_count() or 2)
        cmd = [
            C_FAST_BIN, q_fa, r_fa, prefix_path,
            str(min_len), str(min_sim), str(wl), '1', threads, str(fl),
        ]
    else:
        cmd = [
            GECKO_BIN, 'compare', q_fa, r_fa,
            '-o', outdir, '--prefix', tag,
            '--word-length', str(wl),
            '-l', str(min_len),
            '-s', str(min_sim),
            '--freq-limit', str(fl),
            '-m', mem_budget,
            '--backend', backend,
            '--format', 'frags',
            '--json-stats',
        ]
    with tempfile.TemporaryFile() as out_f, tempfile.TemporaryFile() as err_f:
        t0 = time.perf_counter()
        p = subprocess.Popen(cmd, stdout=out_f, stderr=err_f)
        _, status, rusage = os.wait4(p.pid, 0)
        wall_s = time.perf_counter() - t0
        out_f.seek(0); err_f.seek(0)
        stdout = out_f.read().decode('utf-8', errors='replace')
        stderr = err_f.read().decode('utf-8', errors='replace')
        rss_mib = rusage.ru_maxrss / 1024.0

    if status != 0:
        shutil.rmtree(outdir, ignore_errors=True)
        err_lines = (stderr.strip() or stdout.strip()).splitlines()
        return {'ok': False, 'error': err_lines[-1] if err_lines else f'exit {status}'}, None

    frags_path = os.path.join(outdir, f'{tag}.frags')
    frags = parse_frags(frags_path) if check_parity else None

    if backend == 'c_fast':
        combined = stdout + '\n' + stderr
        m_seq = re.search(r'SeqX:\s*(\d+)\s*bp.*SeqY:\s*(\d+)\s*bp', combined)
        m_frag = re.search(r'Fragments:\s*(\d+)\s*\(F:(\d+)\s*R:(\d+)\)', combined)
        m_time = re.search(r'Timing:\s*load=([0-9.]+)s\s+index=([0-9.]+)s\s+query\+ext=([0-9.]+)s\s+write=([0-9.]+)s', combined)
        load_s = float(m_time.group(1)) if m_time else 0.0
        compute_s = (float(m_time.group(2)) + float(m_time.group(3))) if m_time else wall_s
        write_s = float(m_time.group(4)) if m_time else 0.0
        n_frags = int(m_frag.group(1)) if m_frag else (len(frags) if frags is not None else 0)
        shutil.rmtree(outdir, ignore_errors=True)
        return {
            'ok': True,
            'backend_label': f'c_fast (C -march=native, {os.cpu_count() or 2}t)',
            'wall_s': round(wall_s, 3),
            'load_s': round(load_s, 3),
            'compute_s': round(compute_s, 3),
            'write_s': round(write_s, 3),
            'rss_mib': round(rss_mib, 1),
            'temp_disk_mib': 0.0,
            'total_written_mib': 0.0,
            'fragments': n_frags,
            'forward_fragments': int(m_frag.group(2)) if m_frag else 0,
            'reverse_fragments': int(m_frag.group(3)) if m_frag else 0,
            'query_bases': int(m_seq.group(1)) if m_seq else 0,
            'reference_bases': int(m_seq.group(2)) if m_seq else 0,
        }, frags

    st = json.loads(stdout)
    shutil.rmtree(outdir, ignore_errors=True)
    return {
        'ok': True,
        'backend_label': st.get('backend', backend),
        'wall_s': round(wall_s, 3),
        'load_s': round(st.get('load_secs', 0.0), 3),
        'compute_s': round(st.get('compute_secs', wall_s), 3),
        'write_s': round(st.get('write_secs', 0.0), 3),
        'rss_mib': round(rss_mib, 1),
        'temp_disk_mib': 0.0,
        'total_written_mib': 0.0,
        'fragments': st['fragments'],
        'forward_fragments': st['forward_fragments'],
        'reverse_fragments': st['reverse_fragments'],
        'spilled_runs': st.get('spilled_runs', 0),
        'query_bases': st['query_bases'],
        'reference_bases': st['reference_bases'],
    }, frags

print('Benchmark harness ready.')


In [ ]:
# 5. EXPERIMENT 1: Direct Replication of Torreno & Trelles (2015) Table 2 (All 5 Pairwise Datasets × All 5 Backends)
import csv, json, os

EXP1_JSON = os.path.join(OUT_DIR, 'exp1_original_paper_pairwise.json')
exp1_results = json.load(open(EXP1_JSON)) if (RESUME and os.path.exists(EXP1_JSON)) else {}

if not RUN_EXP1_ORIGINAL_PAIRWISE:
    print('Skipping Experiment 1 (RUN_EXP1_ORIGINAL_PAIRWISE = False).')
else:
    for idx, bm in enumerate(ORIG_PAIRWISE_BENCHMARKS, 1):
        bid = bm['id']
        existing = exp1_results.get(bid, {})
        needed = [b for b in EXP1_BACKENDS if b not in existing]
        if not needed:
            print(f"[{idx}/5] {bm['label']}: already complete, skipping.")
            continue

        print(f"\n[{idx}/5] {bm['label']} (WL={bm['wl']}, L={bm['l']}, S={bm['s']}, FL={bm['fl']})", flush=True)
        q_fa, q_bp = fetch_ncbi_fasta(bm['q_acc'], os.path.join(DATA_DIR, f"{bm['q_name']}_{bm['q_acc']}.fa"))
        r_fa, r_bp = fetch_ncbi_fasta(bm['r_acc'], os.path.join(DATA_DIR, f"{bm['r_name']}_{bm['r_acc']}.fa"))

        run_list = list(needed)
        if 'cpu' in run_list:
            run_list.remove('cpu')
            run_list = ['cpu'] + run_list
        elif CHECK_PARITY and any(b != 'cpu' for b in run_list):
            run_list = ['cpu'] + run_list

        cpu_frags = None
        row = dict(existing)
        for b in run_list:
            rec, frags = run_backend(b, f'exp1_{bid}', q_fa, r_fa, bm['wl'], bm['l'], bm['s'], bm['fl'], check_parity=CHECK_PARITY)
            if b == 'cpu' and rec['ok']:
                cpu_frags = frags
            elif CHECK_PARITY and rec['ok'] and cpu_frags is not None and frags is not None:
                ex_pct, rec_pct = compute_parity(cpu_frags, frags)
                rec['exact_pct'] = round(ex_pct, 3)
                rec['recall_pct'] = round(rec_pct, 4)
                if b == 'legacy_v1':
                    _, v1_rec = compute_parity(frags, cpu_frags)
                    rec['cpu_recovers_v1_pct'] = round(v1_rec, 4)

            if b in needed:
                row[b] = rec

            if rec['ok']:
                pstr = f" | recall={rec['recall_pct']:.3f}%" if 'recall_pct' in rec else ''
                if 'cpu_recovers_v1_pct' in rec:
                    pstr += f" (v2 covers v1={rec['cpu_recovers_v1_pct']:.3f}%)"
                dstr = f" | tmp={rec['temp_disk_mib']:,.1f} MiB (wr={rec['total_written_mib']:,.1f} MiB)" if rec.get('temp_disk_mib', 0) > 0 else ''
                print(
                    f"  [{b:<9}] wall={rec['wall_s']:8.3f}s (comp={rec['compute_s']:7.3f}s) | "
                    f"RSS={rec['rss_mib']:6.1f} MiB | frags={rec['fragments']:,}{dstr}{pstr}",
                    flush=True,
                )
            else:
                print(f"  [{b:<9}] ERROR: {rec['error']}", flush=True)

            exp1_results[bid] = row
            with open(EXP1_JSON, 'w') as f:
                json.dump(exp1_results, f, indent=2)

    # Export CSV, Markdown, and LaTeX Table 1 (Direct 2015 Table 2 Comparison)
    csv_p = os.path.join(OUT_DIR, 'exp1_original_paper_pairwise.csv')
    md_p  = os.path.join(OUT_DIR, 'exp1_original_paper_pairwise.md')
    tex_p = os.path.join(OUT_DIR, 'exp1_original_paper_pairwise.tex')

    fields = ['id', 'label', 'wl', 'l', 's', 'fl', 'backend', 'wall_s', 'load_s', 'compute_s',
              'rss_mib', 'temp_disk_mib', 'total_written_mib', 'fragments', 'exact_pct', 'recall_pct',
              'cpu_recovers_v1_pct', 'speedup_vs_v1_local', 'speedup_vs_v1_2015']
    rows_flat = []
    for bm in ORIG_PAIRWISE_BENCHMARKS:
        row = exp1_results.get(bm['id'], {})
        v1_loc = row.get('legacy_v1', {}).get('wall_s') if row.get('legacy_v1', {}).get('ok') else None
        v1_pub = bm['pub_2015_s']['GECKO_v1_2015']
        for b in ['legacy_v1', 'c_fast', 'cpu', 'wgpu', 'cuda']:
            r = row.get(b, {})
            if not r.get('ok'):
                continue
            rows_flat.append({
                'id': bm['id'], 'label': bm['label'], 'wl': bm['wl'], 'l': bm['l'], 's': bm['s'], 'fl': bm['fl'],
                'backend': b, 'wall_s': r['wall_s'], 'load_s': r.get('load_s', 0), 'compute_s': r.get('compute_s', r['wall_s']),
                'rss_mib': r['rss_mib'], 'temp_disk_mib': r.get('temp_disk_mib', 0.0),
                'total_written_mib': r.get('total_written_mib', 0.0), 'fragments': r['fragments'],
                'exact_pct': r.get('exact_pct', 100.0 if b == 'cpu' else ''),
                'recall_pct': r.get('recall_pct', 100.0 if b == 'cpu' else ''),
                'cpu_recovers_v1_pct': r.get('cpu_recovers_v1_pct', ''),
                'speedup_vs_v1_local': round(v1_loc / r['wall_s'], 2) if (v1_loc and r['wall_s'] > 0) else '',
                'speedup_vs_v1_2015': round(v1_pub / r['wall_s'], 2) if (v1_pub and r['wall_s'] > 0) else '',
            })

    with open(csv_p, 'w', newline='') as f:
        w = csv.DictWriter(f, fieldnames=fields)
        w.writeheader(); w.writerows(rows_flat)

    md_lines = [
        '# Experiment 1: Direct Replication of Torreno & Trelles (2015) Table 2\n',
        '| Dataset | 2015 GECKO v1 (pub) | `legacy_v1` (Colab) | `c_fast` | `cpu` (Rust v2) | `wgpu` | `cuda` | Speedup `cuda` vs `legacy_v1` | `legacy_v1` Temp Disk |',
        '|---|---:|---:|---:|---:|---:|---:|---:|---:|',
    ]
    tex_lines = [
        r'\begin{table*}[t]', r'\centering',
        r'\caption{Direct replication of the five pairwise benchmarks from Torreno \& Trelles (2015) Table 2 across 4 kb--249 Mb. All GECKO v2 backends write 0 MiB of temporary disk.}',
        r'\label{tab:orig_pairwise}',
        r'\begin{tabular}{lrrrrrrr}', r'\toprule',
        r'Dataset & 2015 v1 (s) & \texttt{legacy\_v1} (s) & \texttt{c\_fast} (s) & \texttt{cpu} (s) & \texttt{cuda} (s) & Speedup (vs v1) & v1 Disk (MiB) \\',
        r'\midrule',
    ]
    for bm in ORIG_PAIRWISE_BENCHMARKS:
        row = exp1_results.get(bm['id'], {})
        def gs(b):
            r = row.get(b, {})
            return f"{r['wall_s']:.2f}s" if r.get('ok') else '—'
        def gn(b):
            r = row.get(b, {})
            return f"{r['wall_s']:.2f}" if r.get('ok') else '--'
        v1w = row.get('legacy_v1', {}).get('wall_s') if row.get('legacy_v1', {}).get('ok') else None
        cuw = row.get('cuda', {}).get('wall_s') if row.get('cuda', {}).get('ok') else None
        sp  = f"**{v1w/cuw:.1f}×**" if (v1w and cuw and cuw > 0) else '—'
        sp_t = f"{v1w/cuw:.1f}$\\times$" if (v1w and cuw and cuw > 0) else '--'
        v1d = row.get('legacy_v1', {}).get('temp_disk_mib', 0.0) if row.get('legacy_v1', {}).get('ok') else 0.0
        pub = bm['pub_2015_s']['GECKO_v1_2015']
        md_lines.append(f"| {bm['label']} | {pub:,.2f}s | {gs('legacy_v1')} | {gs('c_fast')} | {gs('cpu')} | {gs('wgpu')} | {gs('cuda')} | {sp} | {v1d:,.1f} MiB |")
        tex_lines.append(f"{bm['label'].replace('_', r'\_')} & {pub:,.2f} & {gn('legacy_v1')} & {gn('c_fast')} & {gn('cpu')} & {gn('cuda')} & {sp_t} & {v1d:,.0f} \\\\")
    tex_lines += [r'\bottomrule', r'\end{tabular}', r'\end{table*}']

    open(md_p, 'w').write('\n'.join(md_lines) + '\n')
    open(tex_p, 'w').write('\n'.join(tex_lines) + '\n')
    print('\n' + '\n'.join(md_lines))


In [ ]:
# 6. EXPERIMENT 2: Original 2015 Table 4 — 10-Species Mammalian Chr1 All-vs-All (45 Pairs!)
#    In Torreno & Trelles (2015), all 45 pairs took 142,954 s (39.7 hours) on GECKO v1
#    and 6.5 - 55 days on MUMmer / Gepard / LASTZ. Here we run ALL 45 pairs on T4 CUDA (~4-6 min)
#    plus the 9 HS x species pairs (Fig. 2 of 2015 paper) on Rust CPU (~3 min)!
import csv, json, os

EXP2_JSON = os.path.join(OUT_DIR, 'exp2_all_vs_all_45pairs.json')
exp2_results = json.load(open(EXP2_JSON)) if (RESUME and os.path.exists(EXP2_JSON)) else {}

if not RUN_EXP2_ALL_VS_ALL_45:
    print('Skipping Experiment 2 (RUN_EXP2_ALL_VS_ALL_45 = False).')
else:
    print('Ensuring all 10 mammalian chr1 FASTAs from 2015 Table 1 are cached locally...', flush=True)
    chr1_paths = {}
    for code, sp, acc in MAMMALIAN_CHR1_10:
        fa, bp = fetch_ncbi_fasta(acc, os.path.join(DATA_DIR, f'{code}_chr1_{acc}.fa'))
        chr1_paths[code] = (fa, bp, sp, acc)

    pairs = []
    codes = [c for c, _, _ in MAMMALIAN_CHR1_10]
    for i in range(len(codes)):
        for j in range(i + 1, len(codes)):
            pairs.append((codes[i], codes[j]))

    print(f'\nRunning {len(pairs)} mammalian chr1 pairs (WL=32, L=1000, S=75, FL={DEFAULT_FL})...', flush=True)
    for idx, (c1, c2) in enumerate(pairs, 1):
        key = f'{c1}-{c2}'
        existing = exp2_results.get(key, {})
        backends_for_pair = ['cuda']
        if RUN_EXP2_ALL_45_ON_CPU or c1 == 'HS':
            backends_for_pair.append('cpu')

        needed = [b for b in backends_for_pair if b not in existing]
        if not needed:
            continue

        q_fa = chr1_paths[c1][0]
        r_fa = chr1_paths[c2][0]
        row = dict(existing)
        for b in needed:
            rec, _ = run_backend(b, f'exp2_{key}', q_fa, r_fa, wl=32, min_len=1000, min_sim=75, fl=DEFAULT_FL, check_parity=False)
            row[b] = rec
            if rec['ok']:
                print(
                    f"  [{idx:2d}/{len(pairs)}] {key:<7} [{b:<4}] wall={rec['wall_s']:6.2f}s "
                    f"(comp={rec['compute_s']:5.2f}s) | RSS={rec['rss_mib']:6.1f} MiB | frags={rec['fragments']:,}",
                    flush=True,
                )
            else:
                print(f"  [{idx:2d}/{len(pairs)}] {key:<7} [{b:<4}] ERROR: {rec['error']}", flush=True)
            exp2_results[key] = row
            with open(EXP2_JSON, 'w') as f:
                json.dump(exp2_results, f, indent=2)

    # Summarize & export CSV / Markdown / LaTeX
    cuda_ok = [exp2_results[f'{a}-{b}']['cuda'] for a, b in pairs if exp2_results.get(f'{a}-{b}', {}).get('cuda', {}).get('ok')]
    cuda_total_s = sum(r['wall_s'] for r in cuda_ok)
    cuda_comp_s  = sum(r['compute_s'] for r in cuda_ok)
    cuda_max_rss = max((r['rss_mib'] for r in cuda_ok), default=0.0)
    total_frags  = sum(r['fragments'] for r in cuda_ok)

    csv_p = os.path.join(OUT_DIR, 'exp2_all_vs_all_45pairs.csv')
    with open(csv_p, 'w', newline='') as f:
        w = csv.DictWriter(f, fieldnames=['pair', 'q', 'r', 'cuda_wall_s', 'cuda_compute_s', 'cuda_rss_mib', 'cuda_frags', 'cpu_wall_s', 'cpu_compute_s', 'cpu_rss_mib', 'cpu_vs_cuda_speedup'])
        w.writeheader()
        for a, b in pairs:
            k = f'{a}-{b}'
            cu = exp2_results.get(k, {}).get('cuda', {})
            cp = exp2_results.get(k, {}).get('cpu', {})
            w.writerow({
                'pair': k, 'q': a, 'r': b,
                'cuda_wall_s': cu.get('wall_s', '') if cu.get('ok') else '',
                'cuda_compute_s': cu.get('compute_s', '') if cu.get('ok') else '',
                'cuda_rss_mib': cu.get('rss_mib', '') if cu.get('ok') else '',
                'cuda_frags': cu.get('fragments', '') if cu.get('ok') else '',
                'cpu_wall_s': cp.get('wall_s', '') if cp.get('ok') else '',
                'cpu_compute_s': cp.get('compute_s', '') if cp.get('ok') else '',
                'cpu_rss_mib': cp.get('rss_mib', '') if cp.get('ok') else '',
                'cpu_vs_cuda_speedup': round(cp['wall_s']/cu['wall_s'], 2) if (cp.get('ok') and cu.get('ok') and cu['wall_s'] > 0) else '',
            })

    md_lines = [
        '# Experiment 2: 10-Species Mammalian Chr1 All-vs-All (45 Pairs, Exact 2015 Table 4 Dataset)\n',
        f'Completed **{len(cuda_ok)}/45 pairs** on GECKO v2 CUDA in **{cuda_total_s:.1f} s ({cuda_total_s/60:.2f} min)** total wall time (compute={cuda_comp_s:.1f} s, peak RSS={cuda_max_rss:.1f} MiB, 0 MiB temp disk, {total_frags:,} total HSPs).\n',
        '| Method (45 mammalian chr1 pairs) | Total Wall Time | Equivalent | Speedup of GECKO v2 CUDA |',
        '|---|---:|---:|---:|',
        f"| **GECKO v2 (`cuda`, T4 GPU)** | **{cuda_total_s:,.1f} s** | **{cuda_total_s/60:.2f} min** | **1.0×** |",
    ]
    for name, sec in [
        ('GECKO v1 (2015, cached dicts)', PUB_2015_TABLE4_TOTALS['GECKO_v1_2015_cached']),
        ('GECKO v1 (2015, uncached)',     PUB_2015_TABLE4_TOTALS['GECKO_v1_2015_nocache']),
        ('MUMmer (2015, 2 TB HPC node)',  PUB_2015_TABLE4_TOTALS['MUMmer_2015']),
        ('Gepard (2015, 2 TB HPC node)',  PUB_2015_TABLE4_TOTALS['Gepard_2015']),
        ('LASTZ (2015)',                  PUB_2015_TABLE4_TOTALS['LASTZ_2015']),
    ]:
        sp = f"**{sec/cuda_total_s:,.0f}×**" if cuda_total_s > 0 else '—'
        hrs = sec / 3600.0
        eq = f'{hrs:.1f} h' if hrs < 48 else f'{hrs/24:.1f} days'
        md_lines.append(f'| {name} | {sec:,.0f} s | {eq} | {sp} |')

    md_lines += [
        '\n### Human chr1 vs. 9 Mammals (Same-Machine Rust `cpu` vs. `cuda`, Fig. 2 of 2015 Paper)\n',
        '| Pair | 2015 GECKO v1 (s) | GECKO v2 `cpu` (s) | GECKO v2 `cuda` (s) | `cuda` vs `cpu` | HSPs |',
        '|---|---:|---:|---:|---:|---:|',
    ]
    for c2 in codes[1:]:
        k = f'HS-{c2}'
        cu = exp2_results.get(k, {}).get('cuda', {})
        cp = exp2_results.get(k, {}).get('cpu', {})
        p15 = PUB_2015_TABLE4_HS_ROW.get(c2, 0)
        sp_cc = f"{cp['wall_s']/cu['wall_s']:.2f}×" if (cp.get('ok') and cu.get('ok') and cu['wall_s'] > 0) else '—'
        md_lines.append(
            f"| HS × {c2} | {p15:,} | {cp.get('wall_s', 0):.2f} | {cu.get('wall_s', 0):.2f} | {sp_cc} | {cu.get('fragments', 0):,} |"
        )
    open(os.path.join(OUT_DIR, 'exp2_all_vs_all_45pairs.md'), 'w').write('\n'.join(md_lines) + '\n')
    print('\n' + '\n'.join(md_lines))


In [ ]:
# 7. EXPERIMENT 3: Full 24-Chromosome Human (GRCh38.p14) vs. Chimp (Clint_PTRv2) Whole Genome
#    Fast in-memory backends ONLY (`c_fast`, `cpu`, `wgpu`, `cuda` — no slow `legacy_v1`!)
import csv, gzip, json, os, shutil, subprocess

EXP3_JSON = os.path.join(OUT_DIR, 'exp3_human_vs_chimp_24chr.json')
exp3_results = json.load(open(EXP3_JSON)) if (RESUME and os.path.exists(EXP3_JSON)) else {}

CHROMOSOMES = [f'chr{i}' for i in range(1, 23)] + ['chrX', 'chrY']
HS_BASE = (
    'https://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/000/001/405/'
    'GCF_000001405.40_GRCh38.p14/GCF_000001405.40_GRCh38.p14_assembly_structure/'
    'Primary_Assembly/assembled_chromosomes/FASTA'
)
PT_BASE = (
    'https://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/002/880/755/'
    'GCF_002880755.1_Clint_PTRv2/GCF_002880755.1_Clint_PTRv2_assembly_structure/'
    'Primary_Assembly/assembled_chromosomes/FASTA'
)

def curl_dl(url, dst):
    subprocess.run([
        'curl', '-fsSL', '--connect-timeout', '15', '--speed-time', '20', '--speed-limit', '50000',
        '--retry', '5', '--retry-delay', '2', '--retry-all-errors', '-o', dst, url,
    ], check=True)

def gunzip_to(src_gz, dst_fa, mode='wb'):
    with gzip.open(src_gz, 'rb') as fin, open(dst_fa, mode) as fout:
        shutil.copyfileobj(fin, fout, 1 << 20)

def download_hs_pt_pair(ch):
    hs_fa = os.path.join(WORK_DIR, f'hs_{ch}.fna')
    pt_fa = os.path.join(WORK_DIR, f'pt_{ch}.fna')
    curl_dl(f'{HS_BASE}/{ch}.fna.gz', hs_fa + '.gz')
    gunzip_to(hs_fa + '.gz', hs_fa, 'wb'); os.remove(hs_fa + '.gz')
    if ch == 'chr2':
        for part, mode in [('chr2A', 'wb'), ('chr2B', 'ab')]:
            curl_dl(f'{PT_BASE}/{part}.fna.gz', os.path.join(WORK_DIR, f'pt_{part}.fna.gz'))
            gunzip_to(os.path.join(WORK_DIR, f'pt_{part}.fna.gz'), pt_fa, mode)
            os.remove(os.path.join(WORK_DIR, f'pt_{part}.fna.gz'))
    else:
        curl_dl(f'{PT_BASE}/{ch}.fna.gz', pt_fa + '.gz')
        gunzip_to(pt_fa + '.gz', pt_fa, 'wb'); os.remove(pt_fa + '.gz')
    return hs_fa, pt_fa

if not RUN_EXP3_WHOLE_GENOME_24CHR:
    print('Skipping Experiment 3 (RUN_EXP3_WHOLE_GENOME_24CHR = False).')
else:
    for idx, ch in enumerate(CHROMOSOMES, 1):
        existing = exp3_results.get(ch, {})
        needed = [b for b in EXP3_BACKENDS if b not in existing]
        if not needed:
            print(f'[{idx:2d}/24] {ch}: already complete, skipping.')
            continue
        print(f'\n[{idx:2d}/24] {ch} — downloading GRCh38 × Clint_PTRv2...', flush=True)
        hs_fa, pt_fa = download_hs_pt_pair(ch)
        run_list = list(needed)
        if 'cpu' in run_list:
            run_list.remove('cpu'); run_list = ['cpu'] + run_list
        elif CHECK_PARITY and any(b != 'cpu' for b in run_list):
            run_list = ['cpu'] + run_list

        cpu_frags = None
        row = dict(existing)
        for b in run_list:
            rec, frags = run_backend(b, f'exp3_{ch}', hs_fa, pt_fa, wl=32, min_len=1000, min_sim=75, fl=DEFAULT_FL, check_parity=CHECK_PARITY)
            if b == 'cpu' and rec['ok']:
                cpu_frags = frags
            elif CHECK_PARITY and rec['ok'] and cpu_frags is not None and frags is not None:
                ex_pct, rec_pct = compute_parity(cpu_frags, frags)
                rec['exact_pct'] = round(ex_pct, 3); rec['recall_pct'] = round(rec_pct, 4)
            if b in needed:
                row[b] = rec
            if rec['ok']:
                pstr = f" | recall={rec['recall_pct']:.3f}%" if 'recall_pct' in rec else ''
                print(f"  [{b:<6}] wall={rec['wall_s']:6.2f}s (comp={rec['compute_s']:5.2f}s) | RSS={rec['rss_mib']:6.1f} MiB | frags={rec['fragments']:,}{pstr}", flush=True)
            else:
                print(f"  [{b:<6}] ERROR: {rec['error']}", flush=True)
            exp3_results[ch] = row
            with open(EXP3_JSON, 'w') as f:
                json.dump(exp3_results, f, indent=2)
        os.remove(hs_fa); os.remove(pt_fa)

    # Export CSV, MD, LaTeX summary
    csv_p = os.path.join(OUT_DIR, 'exp3_human_vs_chimp_24chr.csv')
    with open(csv_p, 'w', newline='') as f:
        w = csv.DictWriter(f, fieldnames=['chr', 'backend', 'wall_s', 'load_s', 'compute_s', 'rss_mib', 'fragments', 'exact_pct', 'recall_pct'])
        w.writeheader()
        for ch in CHROMOSOMES:
            for b in EXP3_BACKENDS:
                r = exp3_results.get(ch, {}).get(b, {})
                if r.get('ok'):
                    w.writerow({
                        'chr': ch, 'backend': b, 'wall_s': r['wall_s'], 'load_s': r.get('load_s', 0),
                        'compute_s': r.get('compute_s', r['wall_s']), 'rss_mib': r['rss_mib'],
                        'fragments': r['fragments'],
                        'exact_pct': r.get('exact_pct', 100.0 if b == 'cpu' else ''),
                        'recall_pct': r.get('recall_pct', 100.0 if b == 'cpu' else ''),
                    })

    md_lines = [
        '# Experiment 3: Whole-Genome 24-Chromosome Human (GRCh38.p14) × Chimp (Clint_PTRv2)\n',
        '| Backend | Chrs | Total Wall (s) | Total Compute (s) | Wall vs `cpu` | Compute vs `cpu` | Peak RSS (MiB) | Total Frags | Min Recall (%) |',
        '|---|---:|---:|---:|---:|---:|---:|---:|---:|',
    ]
    cpu_wall = sum(exp3_results[c]['cpu']['wall_s'] for c in CHROMOSOMES if exp3_results.get(c, {}).get('cpu', {}).get('ok'))
    cpu_comp = sum(exp3_results[c]['cpu']['compute_s'] for c in CHROMOSOMES if exp3_results.get(c, {}).get('cpu', {}).get('ok'))
    for b in ['c_fast', 'cpu', 'wgpu', 'cuda']:
        recs = [exp3_results[c][b] for c in CHROMOSOMES if exp3_results.get(c, {}).get(b, {}).get('ok')]
        if not recs:
            continue
        tw = sum(r['wall_s'] for r in recs); tc = sum(r['compute_s'] for r in recs)
        mr = max(r['rss_mib'] for r in recs); tf = sum(r['fragments'] for r in recs)
        rc = min((r.get('recall_pct', 100.0) for r in recs), default=100.0)
        md_lines.append(f'| `{b}` | {len(recs)}/24 | {tw:.2f} | {tc:.2f} | {cpu_wall/tw:.2f}× | {cpu_comp/tc:.2f}× | {mr:.1f} | {tf:,} | {rc:.3f}% |')
    open(os.path.join(OUT_DIR, 'exp3_human_vs_chimp_24chr.md'), 'w').write('\n'.join(md_lines) + '\n')
    print('\n' + '\n'.join(md_lines))


In [ ]:
# 8. EXPERIMENT 4: Paper Ablations (4A: Zero-Spill Slab Memory Scaling; 4B: WL × FL Sensitivity Frontier)
import csv, json, os

if not RUN_EXP4_ABLATIONS:
    print('Skipping Experiment 4 (RUN_EXP4_ABLATIONS = False).')
else:
    # --- 4A: Zero-Spill Reference-Slab Memory Budget on HS chr1 × PT chr1 (249 Mb × 228 Mb) ---
    print('=== Ablation 4A: Zero-Spill Reference-Slab Memory Scaling (HS chr1 × PT chr1) ===', flush=True)
    hs1, _ = fetch_ncbi_fasta('NC_000001.11', os.path.join(DATA_DIR, 'HS_chr1_NC_000001.11.fa'))
    pt1, _ = fetch_ncbi_fasta('NC_006468.3',  os.path.join(DATA_DIR, 'PT_chr1_NC_006468.3.fa'))

    slab_rows = []
    ref_frags = None
    for b in ['cpu', 'cuda']:
        for mem in ['512M', '1G', '2G', '4G']:
            rec, frags = run_backend(b, f'ab4a_{b}_{mem}', hs1, pt1, wl=32, min_len=1000, min_sim=75, fl=DEFAULT_FL, mem_budget=mem, check_parity=True)
            if not rec['ok']:
                print(f'  [{b:<4}] -m {mem:<4}: ERROR {rec["error"]}')
                continue
            if ref_frags is None:
                ref_frags = frags
                ex_p, rec_p = 100.0, 100.0
            else:
                ex_p, rec_p = compute_parity(ref_frags, frags)
            slab_rows.append({
                'backend': b, 'mem_budget': mem, 'wall_s': rec['wall_s'], 'compute_s': rec['compute_s'],
                'rss_mib': rec['rss_mib'], 'temp_disk_mib': 0.0, 'fragments': rec['fragments'],
                'exact_pct': round(ex_p, 4), 'recall_pct': round(rec_p, 4),
            })
            print(f"  [{b:<4}] -m {mem:<4} | wall={rec['wall_s']:6.2f}s (comp={rec['compute_s']:5.2f}s) | RSS={rec['rss_mib']:6.1f} MiB | frags={rec['fragments']:,} | exact={ex_p:.3f}%", flush=True)

    with open(os.path.join(OUT_DIR, 'exp4a_slab_memory.csv'), 'w', newline='') as f:
        w = csv.DictWriter(f, fieldnames=['backend', 'mem_budget', 'wall_s', 'compute_s', 'rss_mib', 'temp_disk_mib', 'fragments', 'exact_pct', 'recall_pct'])
        w.writeheader(); w.writerows(slab_rows)

    # --- 4B: Dense Short-Seed & High-Frequency Frontier on Drosophila 2R × 3 (20.9 Mb × 19.6 Mb) ---
    print('\n=== Ablation 4B: (WL, FL) Sensitivity Frontier (D. mel 2R × D. pse 3) ===', flush=True)
    dm_fa, _ = fetch_ncbi_fasta('NT_033778.3', os.path.join(DATA_DIR, 'Dmel_2R_NT_033778.3.fa'))
    dp_fa, _ = fetch_ncbi_fasta('NC_009006.2', os.path.join(DATA_DIR, 'Dpse_3_NC_009006.2.fa'))

    sens_rows = []
    for wl in [12, 16, 24, 32]:
        for fl in [20, 100, 1000]:
            r_cpu, f_cpu = run_backend('cpu',  f'ab4b_cpu_w{wl}_f{fl}',  dm_fa, dp_fa, wl=wl, min_len=100, min_sim=50, fl=fl, check_parity=True)
            r_gpu, f_gpu = run_backend('cuda', f'ab4b_cuda_w{wl}_f{fl}', dm_fa, dp_fa, wl=wl, min_len=100, min_sim=50, fl=fl, check_parity=True)
            if r_cpu['ok'] and r_gpu['ok']:
                ex_p, rec_p = compute_parity(f_cpu, f_gpu)
                sp_w = r_cpu['wall_s'] / r_gpu['wall_s'] if r_gpu['wall_s'] > 0 else 0.0
                sp_c = r_cpu['compute_s'] / r_gpu['compute_s'] if r_gpu['compute_s'] > 0 else 0.0
                sens_rows.append({
                    'wl': wl, 'fl': fl,
                    'cpu_wall_s': r_cpu['wall_s'], 'cpu_compute_s': r_cpu['compute_s'], 'cpu_rss_mib': r_cpu['rss_mib'],
                    'cuda_wall_s': r_gpu['wall_s'], 'cuda_compute_s': r_gpu['compute_s'], 'cuda_rss_mib': r_gpu['rss_mib'],
                    'wall_speedup': round(sp_w, 2), 'compute_speedup': round(sp_c, 2),
                    'fragments': r_cpu['fragments'], 'exact_pct': round(ex_p, 4), 'recall_pct': round(rec_p, 4),
                })
                print(
                    f"  WL={wl:2d} FL={fl:4d} | cpu={r_cpu['wall_s']:6.2f}s | cuda={r_gpu['wall_s']:5.2f}s "
                    f"({sp_w:5.2f}× wall, {sp_c:5.2f}× comp) | frags={r_cpu['fragments']:,} | recall={rec_p:.3f}%",
                    flush=True,
                )

    with open(os.path.join(OUT_DIR, 'exp4b_wl_fl_sensitivity.csv'), 'w', newline='') as f:
        w = csv.DictWriter(f, fieldnames=['wl', 'fl', 'cpu_wall_s', 'cpu_compute_s', 'cpu_rss_mib', 'cuda_wall_s', 'cuda_compute_s', 'cuda_rss_mib', 'wall_speedup', 'compute_speedup', 'fragments', 'exact_pct', 'recall_pct'])
        w.writeheader(); w.writerows(sens_rows)

    print('\nAblations 4A & 4B saved to:', OUT_DIR)


In [ ]:
# 9. Generate Publication-Ready 300-DPI Multi-Panel Figures (PNG + PDF) & 1-Click ZIP Bundle
import csv, json, os, zipfile
import matplotlib.pyplot as plt
import numpy as np

plt.rcParams.update({
    'font.size': 10, 'axes.titlesize': 11, 'axes.labelsize': 10,
    'xtick.labelsize': 8.5, 'ytick.labelsize': 8.5, 'legend.fontsize': 8.5,
})

fig, axes = plt.subplots(2, 2, figsize=(14, 9.5))

# --- Panel A: Exp 1 — Original 2015 Table 2 Replication (Log Runtime across 4 kb -> 249 Mb) ---
ax = axes[0, 0]
exp1_p = os.path.join(OUT_DIR, 'exp1_original_paper_pairwise.json')
if os.path.exists(exp1_p):
    e1 = json.load(open(exp1_p))
    short_names = ['TYLCV\n(4 kb)', 'Buchnera\n(0.64 Mb)', 'E. coli\n(5 Mb)', 'Drosophila\n(20 Mb)', 'HS×PT chr1\n(249 Mb)']
    ids = [b['id'] for b in ORIG_PAIRWISE_BENCHMARKS]
    x = np.arange(len(ids))
    styles = [
        ('legacy_v1', 'Legacy GECKO v1 (disk)', '#95a5a6', 's', '--'),
        ('c_fast',    'C gecko_fast (-march=native)', '#e67e22', '^', '-'),
        ('cpu',       'GECKO v2 Rust CPU', '#2980b9', 'o', '-'),
        ('wgpu',      'GECKO v2 WGPU', '#8e44ad', 'D', '-'),
        ('cuda',      'GECKO v2 CUDA (T4)', '#27ae60', '*', '-'),
    ]
    for b, lbl, col, mk, ls in styles:
        ys = [e1.get(i, {}).get(b, {}).get('wall_s') if e1.get(i, {}).get(b, {}).get('ok') else np.nan for i in ids]
        ax.plot(x, ys, marker=mk, linestyle=ls, color=col, linewidth=2.0, markersize=7, label=lbl)
    ax.set_yscale('log')
    ax.set_xticks(x); ax.set_xticklabels(short_names)
    ax.set_ylabel('Wall Time (s, log scale)')
    ax.set_title('(A) Original 2015 Table 2 Replication (4 kb → 249 Mb)')
    ax.grid(True, which='both', alpha=0.25)
    ax.legend(loc='upper left', frameon=True)

# --- Panel B: Exp 2 — 10-Species Mammalian Chr1 All-vs-All (45 Pairs) vs 2015 Table 4 ---
ax = axes[0, 1]
exp2_p = os.path.join(OUT_DIR, 'exp2_all_vs_all_45pairs.json')
if os.path.exists(exp2_p):
    e2 = json.load(open(exp2_p))
    cu_tot = sum(v['cuda']['wall_s'] for v in e2.values() if v.get('cuda', {}).get('ok'))
    labels = ['GECKO v2\nCUDA (T4)', 'GECKO v1\n(2015 cached)', 'MUMmer\n(2015 HPC)', 'Gepard\n(2015 HPC)', 'LASTZ\n(2015)']
    vals   = [cu_tot, PUB_2015_TABLE4_TOTALS['GECKO_v1_2015_cached'], PUB_2015_TABLE4_TOTALS['MUMmer_2015'],
              PUB_2015_TABLE4_TOTALS['Gepard_2015'], PUB_2015_TABLE4_TOTALS['LASTZ_2015']]
    cols   = ['#27ae60', '#95a5a6', '#7f8c8d', '#bdc3c7', '#636e72']
    bars = ax.bar(labels, vals, color=cols, edgecolor='black', linewidth=0.6)
    ax.set_yscale('log')
    ax.set_ylabel('Total Wall Time for 45 Pairs (s, log scale)')
    ax.set_title('(B) 10-Species Mammalian Chr1 All-vs-All (45 Pairs)')
    ax.grid(True, axis='y', which='both', alpha=0.25)
    for bar, v in zip(bars, vals):
        txt = f'{v/60:.1f}m' if v < 3600 else (f'{v/3600:.1f}h' if v < 172800 else f'{v/86400:.1f}d')
        if cu_tot > 0 and v > cu_tot * 1.1:
            txt += f'\n({v/cu_tot:,.0f}×)'
        ax.text(bar.get_x() + bar.get_width()/2, v * 1.25, txt, ha='center', va='bottom', fontsize=8, fontweight='bold')

# --- Panel C: Exp 3 — Whole-Genome 24-Chr Human × Chimp Per-Chromosome Scaling ---
ax = axes[1, 0]
exp3_p = os.path.join(OUT_DIR, 'exp3_human_vs_chimp_24chr.json')
if os.path.exists(exp3_p):
    e3 = json.load(open(exp3_p))
    chs = [f'chr{i}' for i in range(1, 23)] + ['chrX', 'chrY']
    x = np.arange(len(chs))
    for b, lbl, col, mk in [
        ('c_fast', 'C gecko_fast', '#e67e22', '^'),
        ('cpu',    'Rust CPU',     '#2980b9', 'o'),
        ('wgpu',   'WGPU',         '#8e44ad', 'D'),
        ('cuda',   'CUDA (T4)',    '#27ae60', '*'),
    ]:
        ys = [e3.get(c, {}).get(b, {}).get('wall_s') if e3.get(c, {}).get(b, {}).get('ok') else np.nan for c in chs]
        if any(np.isfinite(ys)):
            ax.plot(x, ys, marker=mk, color=col, linewidth=1.6, markersize=4.5, label=lbl)
    ax.set_yscale('log')
    ax.set_xticks(x); ax.set_xticklabels([c.replace('chr', '') for c in chs], rotation=45)
    ax.set_xlabel('Chromosome (Human GRCh38.p14 × Chimp Clint_PTRv2)')
    ax.set_ylabel('Wall Time (s, log scale)')
    ax.set_title('(C) Full 24-Chromosome Whole-Genome Scaling')
    ax.grid(True, which='both', alpha=0.25)
    ax.legend(loc='upper right', frameon=True)

# --- Panel D: Exp 4A — Zero-Spill Reference-Slab Memory Budget (HS chr1 × PT chr1) ---
ax = axes[1, 1]
ab4a_p = os.path.join(OUT_DIR, 'exp4a_slab_memory.csv')
if os.path.exists(ab4a_p):
    rows = list(csv.DictReader(open(ab4a_p)))
    budgets = ['512M', '1G', '2G', '4G']
    xb = np.arange(len(budgets))
    w = 0.35
    ax2 = ax.twinx()
    for i, (b, col) in enumerate([('cpu', '#2980b9'), ('cuda', '#27ae60')]):
        rss = [next((float(r['rss_mib']) for r in rows if r['backend'] == b and r['mem_budget'] == m), np.nan) for m in budgets]
        wt  = [next((float(r['wall_s'])  for r in rows if r['backend'] == b and r['mem_budget'] == m), np.nan) for m in budgets]
        ax.bar(xb + (i - 0.5)*w, rss, width=w, color=col, alpha=0.75, edgecolor='black', linewidth=0.5, label=f'{b} Peak RSS')
        ax2.plot(xb, wt, marker='o', linestyle='--', color=col, linewidth=2.0, label=f'{b} Wall (s)')
    ax.set_xticks(xb); ax.set_xticklabels(budgets)
    ax.set_xlabel('Memory Budget (-m) on HS chr1 × PT chr1 (0 MiB disk spill)')
    ax.set_ylabel('Peak Host RSS (MiB)')
    ax2.set_ylabel('Wall Time (s)')
    ax.set_title('(D) Zero-Spill Reference-Slab Memory Scaling')
    ax.grid(True, axis='y', alpha=0.25)
    h1, l1 = ax.get_legend_handles_labels(); h2, l2 = ax2.get_legend_handles_labels()
    ax.legend(h1 + h2, l1 + l2, loc='upper left', frameon=True, fontsize=7.5)

plt.tight_layout()
png_p = os.path.join(OUT_DIR, 'gecko2_paper_figures.png')
pdf_p = os.path.join(OUT_DIR, 'gecko2_paper_figures.pdf')
plt.savefig(png_p, dpi=300, bbox_inches='tight')
plt.savefig(pdf_p, bbox_inches='tight')
plt.show()

# Bundle ALL paper results into a single 1-click download ZIP
zip_path = '/content/gecko2_paper_results.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fn in sorted(os.listdir(OUT_DIR)):
        zf.write(os.path.join(OUT_DIR, fn), arcname=fn)

print(f'\nSaved 300-DPI figures: {png_p} & {pdf_p}')
print(f'Bundled ALL results ({os.path.getsize(zip_path)/1024:.1f} KiB) into: {zip_path}')
print('Contents:', ', '.join(sorted(os.listdir(OUT_DIR))))
